In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("combined_data_clean.csv")
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)


Columns: ['FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HST', 'AST', 'B365H', 'B365A', 'BWH', 'BWA', 'PSH', 'PSA', 'WHH', 'WHA', 'MaxH', 'MaxA', 'AvgH', 'AvgA', 'AHh', 'B365CH', 'B365CA', 'BWCH', 'PSCH', 'PSCA', 'WHCH', 'MaxCH', 'MaxCA', 'AvgCH', 'AvgCA', 'AHCh', 'IWH', 'IWA', 'VCH', 'VCA']
Shape: (3863, 35)


In [ ]:
target = "FTR"


features = [c for c in df.columns if c != target]

leak_cols = ["FTHG", "FTAG"]
features = [c for c in features if c not in leak_cols]


features = [c for c in features if c in df.columns]

df2 = df.dropna(subset=features + [target]).copy()

X = df2[features]
y = df2[target]

print("Using features:", features)
print("y distribution:\n", y.value_counts())
print("Clean shape:", df2.shape)


Using features: ['HTHG', 'HTAG', 'HST', 'AST', 'B365H', 'B365A', 'BWH', 'BWA', 'PSH', 'PSA', 'WHH', 'WHA', 'MaxH', 'MaxA', 'AvgH', 'AvgA', 'AHh', 'B365CH', 'B365CA', 'BWCH', 'PSCH', 'PSCA', 'WHCH', 'MaxCH', 'MaxCA', 'AvgCH', 'AvgCA', 'AHCh', 'IWH', 'IWA', 'VCH', 'VCA']
y distribution:
 FTR
2    1736
0    1105
1    1022
Name: count, dtype: int64
Clean shape: (3863, 35)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

knn_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=25, weights="distance"))
])

knn_model.fit(X_train, y_train)
pred = knn_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))


Accuracy: 0.6261319534282018


In [ ]:

labels = np.unique(np.concatenate([y_test.to_numpy(), pred]))

print("Labels in test/pred:", labels)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred, labels=labels))
print("\nReport:\n", classification_report(y_test, pred, labels=labels, digits=4))


Labels in test/pred: [0 1 2]

Confusion Matrix:
 [[152  34  35]
 [ 64  50  91]
 [ 29  36 282]]

Report:
               precision    recall  f1-score   support

           0     0.6204    0.6878    0.6524       221
           1     0.4167    0.2439    0.3077       205
           2     0.6912    0.8127    0.7470       347

    accuracy                         0.6261       773
   macro avg     0.5761    0.5815    0.5690       773
weighted avg     0.5981    0.6261    0.6034       773

